### Import packages

In [1]:
import pandas as pd
from scipy.stats import spearmanr
from scipy.stats import chi2_contingency
import numpy as np


### EDA

1. LOAD DATA

In [42]:
# load data
data = pd.read_csv('data/data_merged.csv', index_col=False)
data = data.iloc[:,1:]

/var/folders/cr/df6gvtx90rz5bg2m0clb1gym0000gn/T/ipykernel_51897/2634253260.py:2: DtypeWarning: Columns (42,43,44,45,46,47,48,49) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('data/data_merged.csv', index_col=False)


In [44]:
data.columns

Index(['weight', 'BNPL1', 'BNPL3', 'BNPL4_a', 'BNPL4_b', 'BNPL4_c', 'BNPL4_d',
       'BNPL4_e', 'BNPL4_f', 'EF3_a', 'EF3_b', 'EF3_c', 'EF3_d', 'EF3_e',
       'EF3_f', 'EF3_g', 'EF3_h', 'I20', 'ppfs0596', 'ppeducat', 'ppage',
       'ppagecat', 'ppethm', 'ppgender', 'ppinc7', 'ppemploy', 'ppmarit5',
       'pphhsize', 'ppkid017', 'ppfs1482', 'SL6', 'B3', 'A6', 'C4A', 'EF1',
       'L0C', 'I41_c', 'I41_e', 'A1_a', 'A1_b', 'A1_c', 'X12_a', 'X12_b',
       'X12_c', 'X12_d', 'X12_e', 'X12_f', 'X12_g', 'EF5C', 'year'],
      dtype='object')

2. Check columns

In [47]:
columns = ['BNPL1', 'BNPL3', 'BNPL4_a', 'BNPL4_b', 'BNPL4_c', 'BNPL4_d',
       'BNPL4_e', 'BNPL4_f', 'EF3_a', 'EF3_b', 'EF3_c', 'EF3_d', 'EF3_e',
       'EF3_f', 'EF3_g', 'EF3_h', 'I20', 'ppfs0596', 'ppeducat', 'ppage',
       'ppagecat', 'ppethm', 'ppgender', 'ppinc7', 'ppemploy', 'ppmarit5',
       'pphhsize', 'ppkid017', 'ppfs1482', 'SL6', 'B3', 'A6', 'C4A', 'EF1',
       'L0C', 'I41_c', 'I41_e', 'A1_a', 'A1_b', 'A1_c', 'X12_a', 'X12_b',
  'X12_c', 'X12_d', 'X12_e', 'X12_f', 'X12_g', 'EF5C']


column_df = []
for c in columns:
  column_df.append({
    'column': c,
    'unique values': data[c].unique()
  })

pd.DataFrame(column_df)

,column,unique values
0,BNPL1,"[No, Yes]"
1,BNPL3,"[nan, No, Yes]"
2,BNPL4_a,"[nan, Yes, No]"
3,BNPL4_b,"[nan, Yes, No]"
4,BNPL4_c,"[nan, Yes, No]"
5,BNPL4_d,"[nan, No, Yes]"
6,BNPL4_e,"[nan, Yes, No]"
7,BNPL4_f,"[nan, No, Yes]"
8,EF3_a,[Put it on my credit card and pay it off in fu...
9,EF3_b,"[No, Put it on my credit card and pay it off o..."


3. Create ordinal columns

In [214]:
# map columms from categorial to ordianal
ppfs0596_mapping = {
    'Under $50,000': 1,
    '$50,000 - $99,999': 2,
    '$100,000 - $249,999': 3,
    '$250,000 - $499,999': 4,
    '$500,000 - $999,999': 5,
    '$1,000,000 or more': 6,
}

ppinc7_mapping = {
    'Less than $10,000': 1,
    '$10,000 to $24,999': 2,
    '$25,000 to $49,999': 3,
    '$50,000 to $74,999': 4,
    '$75,000 to $99,999': 5,
    '$100,000 to $149,999': 6,
    '$150,000 or more': 7
}

L0C_mapping = {
  "1": 0,
  "2": 1,
  "3": 3,
  "4":4,
  "5 or more": 5
}

BNPL1_mapping = {
    "Yes": 1,
    "No": 0,
}

BNPL3_mapping = {
    "Yes": 1,
    "No": 0,
}

# A6_mapping = {
#     "Not confident": 1,
#     "Somewhat confident": 2,
#     "Don’t know": 3,
#     "Very confident": 4
# }

# C4A_mapping = {
#     "Never carried an unpaid balance (always pay in full)": 0,
#     "Once": 1,
#     "Some of the time": 2,
#     "Most or all of the time": 3
# }

# ppemploy_mapping = {
#     "Working part-time": 2,
#     "Not working": 3,
#     "Working full-time": 1
# }

# INF4_mapping = {
#     "Much worse": 1,
#     "Somewhat worse": 2,
#     "Little or no effect": 3,
#     "Somewhat better":4,
#     "Much better": 5
# }

# I9_mapping = {
#     "Roughly the same amount each month": 1,
#     "Occasionally varies from month to month": 2,
#     "Varies quite often from month to month": 3
# }


In [ ]:
# convert columns to ordinal
data["ppfs0596_ord"] = data["ppfs0596"].map(ppfs0596_mapping)
data["ppinc7_ord"] = data["ppinc7"].map(ppinc7_mapping)
data["L0C_ord"] = data["L0C"].map(L0C_mapping)
data["BNPL1_ord"] = data["BNPL1"].map(BNPL1_mapping)
data["BNPL3_ord"] = data["BNPL3"].map(BNPL3_mapping)

4. Spearman correlation

4.1 Spearman with BNPL1

In [212]:
# spearman relationship
spearman_BNPL1 = []

variables = ['ppfs0596_ord','ppinc7_ord','ppage','pphhsize','ppkid017','L0C_ord']

for var in variables:
  # spearman correlation
  corr = spearmanr(data[var], data["BNPL1_ord"], nan_policy="omit")[0]
  p = spearmanr(data[var], data["BNPL1_ord"], nan_policy="omit")[1]
  spearman_BNPL1.append({
    "variable": var,
    "corr": corr,
    "p_value": p
    })

# convert to DataFrame for display
spearman_BNPL1_results = pd.DataFrame(spearman_BNPL1)
spearman_BNPL1_results

,variable,corr,p_value
0,ppfs0596_ord,-0.157546,2.101884e-168
1,ppinc7_ord,-0.082642,2.269679e-72
2,ppage,-0.129598,5.249315e-176
3,pphhsize,0.052759,1.775330e-30
4,ppkid017,0.070704,2.049620e-53
5,L0C_ord,0.010696,2.658669e-01


4.2 Spearman with BNPL3

In [216]:
# spearman relationship
spearman_BNPL3 = []

variables = ['ppfs0596_ord','ppinc7_ord','ppage','pphhsize','ppkid017','L0C_ord']

for var in variables:
  # spearman correlation
  corr = spearmanr(data[var], data["BNPL3_ord"], nan_policy="omit")[0]
  p = spearmanr(data[var], data["BNPL3_ord"], nan_policy="omit")[1]
  spearman_BNPL3.append({
    "variable": var,
    "corr": corr,
    "p_value": p
    })

# convert to DataFrame for display
spearman_BNPL3_results = pd.DataFrame(spearman_BNPL3)
spearman_BNPL3_results

,variable,corr,p_value
0,ppfs0596_ord,-0.124677,1.442574e-12
1,ppinc7_ord,-0.206426,4.186553e-54
2,ppage,-0.136674,2.159552e-24
3,pphhsize,0.077398,8.763084e-09
4,ppkid017,0.109885,2.829229e-16
5,L0C_ord,0.089671,1.824879e-04


5. One Hot Encode

In [ ]:
# OHE for multi-class categorical columns
columns=['I20','ppfs0596','ppeducat','ppethm','ppmarit5','ppagecat','ppinc7','ppemploy','ppfs1482','B3','C4A','L0C']
dummies = {}
for c in columns:
    dummies[c] = pd.get_dummies(data[c], prefix=c)

In [ ]:
# add OHE columns to data
data = pd.concat([data, pd.concat(dummies.values(), axis=1)], axis=1)

6. CHI-SQUARE TEST

6.1 va. BNPL1

In [ ]:
# predictors
predictors = ['I20',
              'ppfs0596',
              'ppeducat',
              'ppgender',
              'ppethm',
              'ppmarit5',
              'ppagecat',
              'ppinc7',
              'ppemploy',
              'ppfs1482',
              'SL6','A6', 'B3',
              'C4A','L0C',
              'EF1',
              'A1_a','A1_b','A1_c',
              'X12_a','X12_b','X12_c','X12_d','X12_e','X12_f','X12_g',
              'EF5C'
              ] + [col for df in dummies.values() for col in df.columns]
# BNPL use
    # define cramers_v function
def compute_cramers_v(chi2, table):
    n = table.values.sum()
    r, c = table.shape
    k = min(r, c)
    return np.sqrt(chi2 / (n * (k - 1)))


results = []

for var in predictors:
    # remove na
    subset = data[[var, "BNPL1"]].dropna()

    # contingency table
    table = pd.crosstab(subset[var], subset["BNPL1"])

    # chi-square test
    chi2, p, dof, expected = chi2_contingency(table)

    # cramér's v
    v = compute_cramers_v(chi2, table)

    results.append({
        "variable": var,
        "chi2": chi2,
        "p_value": p,
        "cramers_v": v
    })

# convert to DataFrame for display
BNPL1_chisqr_result = pd.DataFrame(results)
display(BNPL1_chisqr_result[BNPL1_chisqr_result['cramers_v']>=0.15])


,variable,chi2,p_value,cramers_v
1,ppfs0596,785.899053,1.714126e-166,0.157056
9,ppfs1482,2095.860225,0.000000e+00,0.228806
11,A6,1807.429243,0.000000e+00,0.195611
13,C4A,2409.108407,0.000000e+00,0.244392
15,EF1,1395.318197,2.186735e-305,0.171870
16,A1_a,1028.181904,1.344683e-225,0.249273
17,A1_b,774.527515,1.863587e-170,0.216351
18,A1_c,954.507724,1.389561e-209,0.240176
23,X12_e,228.753785,2.122007e-50,0.193255
69,ppfs1482_Excellent,1413.792239,2.115226e-309,0.173004


6.2 va. BNPL3

In [196]:
# predictors
predictors = ['I20',
              'ppfs0596',
              'ppeducat',
              'ppgender',
              'ppethm',
              'ppmarit5',
              'ppagecat',
              'ppinc7',
              'ppemploy',
              'ppfs1482',
              'SL6','A6', 'B3',
              'C4A','L0C',
              'EF1',
              'A1_a','A1_b','A1_c',
              'X12_a','X12_b','X12_c','X12_d','X12_e','X12_f','X12_g',
              'EF5C'
              ] + [col for df in dummies.values() for col in df.columns]

# define cramers_v function
def compute_cramers_v(chi2, table):
    n = table.values.sum()
    r, c = table.shape
    k = min(r, c)
    return np.sqrt(chi2 / (n * (k - 1)))

results = []

for var in predictors:
    # remove na
    subset = data[[var, "BNPL3"]].dropna()

    # contingency table
    table = pd.crosstab(subset[var], subset["BNPL3"])

    # chi-square test
    chi2, p, dof, expected = chi2_contingency(table)

    # cramér's v
    v = compute_cramers_v(chi2, table)

    results.append({
        "variable": var,
        "chi2": chi2,
        "p_value": p,
        "cramers_v": v
    })

# convert to DataFrame for display
BNPL3_chisqr_result = pd.DataFrame(results)
display(BNPL3_chisqr_result[BNPL3_chisqr_result['cramers_v']>=0.15])


,variable,chi2,p_value,cramers_v
0,I20,153.998819,3.627283e-34,0.167164
2,ppeducat,127.669360,1.719082e-27,0.152205
7,ppinc7,248.637921,8.016644e-51,0.212407
9,ppfs1482,414.660244,2.051954e-87,0.309423
10,SL6,109.305151,1.391303e-25,0.268339
11,A6,527.580377,5.026688e-114,0.309406
15,EF1,141.565359,1.210373e-32,0.160274
16,A1_a,293.640910,8.002979e-66,0.301094
17,A1_b,103.896530,2.131625e-24,0.179100
18,A1_c,223.091585,1.914480e-50,0.262444


6. Crosstab results for important predictors

6.1 Crosstab results with BNPL1

In [208]:
BNPL1_crosstab = {}
for var in  BNPL1_chisqr_result[BNPL1_chisqr_result['cramers_v']>=0.15]['variable']:
  subset = data[[var, "BNPL1"]].dropna()
  # crosstab
  BNPL1_crosstab[var] = pd.crosstab(subset[var], subset["BNPL1"], normalize='index')


In [209]:
for k in BNPL1_crosstab.keys():
  display(BNPL1_crosstab[k])

BNPL1,No,Yes
ppfs0596,,
"$1,000,000 or more",0.961734,0.038266
"$100,000 - $249,999",0.928405,0.071595
"$250,000 - $499,999",0.938106,0.061894
"$50,000 - $99,999",0.907576,0.092424
"$500,000 - $999,999",0.948573,0.051427
Not sure,0.907616,0.092384
"Under $50,000",0.840074,0.159926


BNPL1,No,Yes
ppfs1482,,
Don’t know,0.934459,0.065541
Excellent,0.942895,0.057105
Fair,0.740901,0.259099
Good,0.856545,0.143455
Poor,0.752776,0.247224
Very poor,0.812731,0.187269


BNPL1,No,Yes
A6,,
Don’t know,0.891905,0.108095
Not confident,0.752208,0.247792
Somewhat confident,0.816560,0.183440
Very confident,0.923720,0.076280


BNPL1,No,Yes
C4A,,
Most or all of the time,0.764915,0.235085
Never carried an unpaid balance (always pay in full),0.951803,0.048197
Once,0.864422,0.135578
Some of the time,0.835732,0.164268


BNPL1,No,Yes
EF1,,
No,0.817121,0.182879
Yes,0.929345,0.070655


...

BNPL1,No,Yes
A1_c,,
No,0.848165,0.151835
Yes,0.597097,0.402903


BNPL1,No,Yes
X12_e,,
Major concern,0.765704,0.234296
Minor concern,0.873541,0.126459
Not a concern,0.929754,0.070246


BNPL1,No,Yes
ppfs1482_Excellent,,
False,0.831506,0.168494
True,0.942895,0.057105


BNPL1,No,Yes
C4A_Most or all of the time,,
False,0.911207,0.088793
True,0.764915,0.235085


BNPL1,No,Yes
C4A_Never carried an unpaid balance (always pay in full),,
False,0.824094,0.175906
True,0.951803,0.048197


6.2 Crosstab results with BNPL3

In [210]:
BNPL3_crosstab = {}
for var in  BNPL3_chisqr_result[BNPL3_chisqr_result['cramers_v']>=0.15]['variable']:
  subset = data[[var, "BNPL3"]].dropna()
  # crosstab
  BNPL3_crosstab[var] = pd.crosstab(subset[var], subset["BNPL3"], normalize='index')

In [211]:
for k in BNPL3_crosstab.keys():
  display(BNPL3_crosstab[k])

BNPL3,No,Yes
I20,,
Less than your income,0.900272,0.099728
More than your income,0.740157,0.259843
The same as your income,0.824691,0.175309


BNPL3,No,Yes
ppeducat,,
Bachelor's degree or higher,0.893733,0.106267
High school graduate (high school diploma or the equivalent GED),0.787143,0.212857
No high school diploma or GED,0.689904,0.310096
Some college or Associate's degree,0.814516,0.185484


BNPL3,No,Yes
ppinc7,,
"$10,000 to $24,999",0.678819,0.321181
"$100,000 to $149,999",0.901456,0.098544
"$150,000 or more",0.918320,0.081680
"$25,000 to $49,999",0.774587,0.225413
"$50,000 to $74,999",0.820538,0.179462
"$75,000 to $99,999",0.870844,0.129156
"Less than $10,000",0.665339,0.334661


BNPL3,No,Yes
ppfs1482,,
Don’t know,0.748466,0.251534
Excellent,0.931474,0.068526
Fair,0.792731,0.207269
Good,0.895193,0.104807
Poor,0.595745,0.404255
Very poor,0.566502,0.433498


BNPL3,No,Yes
SL6,,
No,0.817032,0.182968
Yes,0.536145,0.463855


...

BNPL3,No,Yes
X12_c,,
Major concern,0.686111,0.313889
Minor concern,0.764706,0.235294
Not a concern,0.880531,0.119469


BNPL3,No,Yes
X12_e,,
Major concern,0.648794,0.351206
Minor concern,0.834615,0.165385
Not a concern,0.896552,0.103448


BNPL3,No,Yes
EF5C,,
No,0.618799,0.381201
Yes,0.868588,0.131412


BNPL3,No,Yes
ppfs1482_Excellent,,
False,0.792998,0.207002
True,0.931474,0.068526


BNPL3,No,Yes
ppfs1482_Poor,,
False,0.843553,0.156447
True,0.595745,0.404255
